In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_name = df[
    (df["ISSUE"].str.strip() == "Name") &
    (df["Network ID"] != 11)
]
df_name_new =  df_name[['Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)


df_name_new

,Station ID,Unique Names,Native ID
0,1694.0,TUMBLER(DENISON) -> TUMBLER HUB,127
1,1727.0,EUTSUK,164
2,1769.0,PLACE LK. -> PLACE LAKE,212
3,1891.0,SALMON ARM,346
4,1908.0,REVELSTOKE FS -> REVELSTOKE,363
...,...,...,...
62,12349.0,Site C North Camp,SNC
63,2505.0,Stave R. ab Stave Lk -> Stave River above Stav...,STA
64,2502.0,Sugar Lake Res. @ Outlet -> Sigar Lake Resevoi...,SGL
65,2507.0,Wahleach (Jones) Res. -> Wahleach Resevoir (Jo...,WAH


In [5]:
def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_name = df_name_new.copy()

df_name[['old_name', 'new_name', 'n_names']] = (
    df_name['Unique Names'].apply(split_station_name)
)

df_name = df_name.drop(columns= 'Unique Names')

In [6]:
df_name

,Station ID,Native ID,old_name,new_name,n_names
0,1694.0,127,TUMBLER(DENISON),TUMBLER HUB,2
1,1727.0,164,EUTSUK,EUTSUK,1
2,1769.0,212,PLACE LK.,PLACE LAKE,2
3,1891.0,346,SALMON ARM,SALMON ARM,1
4,1908.0,363,REVELSTOKE FS,REVELSTOKE,2
...,...,...,...,...,...
62,12349.0,SNC,Site C North Camp,Site C North Camp,1
63,2505.0,STA,Stave R. ab Stave Lk,Stave River above Stave Lake,2
64,2502.0,SGL,Sugar Lake Res. @ Outlet,Sigar Lake Resevoir at the Outlet,2
65,2507.0,WAH,Wahleach (Jones) Res.,Wahleach Resevoir (Jones Lake),2


In [7]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name
FROM meta_history
WHERE station_id = :station_id
""")

In [8]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 1694, TUMBLER(DENISON) the names match 
✅ Station 1727, EUTSUK the names match 
✅ Station 1769, PLACE LK. the names match 
✅ Station 1891, SALMON ARM the names match 
✅ Station 1908, REVELSTOKE FS the names match 
✅ Station 1926, FALLS CK the names match 
✅ Station 1927, HOWSER the names match 
✅ Station 1934, 8 MILE the names match 
✅ Station 1940, CRESTON the names match 
✅ Station 1957, TOBY the names match 
✅ Station 2236, NEGRO CK the names match 
✅ Station 2302, TURNER _Tweedsmuir the names match 
✅ Station 2304, GOLDSTREAM II the names match 
✅ Station 2311, ANDERSON the names match 
✅ Station 2328, OELRICH the names match 
✅ Station 2332, SASKUM the names match 
✅ Station 11001, NORTH SLOQUET the names match 
✅ Station 12018, NARROWS INLET the names match 
✅ Station 12337, INKANEEP EXT the names match 
✅ Station 12524, APEX EXT the names match 
✅ Station 12622, MERRITT 2 HUB the names match 
✅ Station 12350, 85th Ave the names match 
✅ Station 12343, Attachie Flat Low

In [9]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 1694: (TUMBLER(DENISON)) → (TUMBLER HUB)
✅ Updated station 1727: (EUTSUK) → (EUTSUK)
✅ Updated station 1769: (PLACE LK.) → (PLACE LAKE)
✅ Updated station 1891: (SALMON ARM) → (SALMON ARM)
✅ Updated station 1908: (REVELSTOKE FS) → (REVELSTOKE)
✅ Updated station 1926: (FALLS CK) → (FALLS CREEK)
✅ Updated station 1927: (HOWSER) → (HOWSER)
✅ Updated station 1934: (8 MILE) → (EIGHT MILE)
✅ Updated station 1940: (CRESTON) → (CRESTON)
✅ Updated station 1957: (TOBY) → (TOBY HUB)
✅ Updated station 2236: (NEGRO CK) → (Bigattini)
✅ Updated station 2302: (TURNER _Tweedsmuir) → (TURNER _Tweedsmuir)
✅ Updated station 2304: (GOLDSTREAM II) → (GOLDSTREAM 2)
✅ Updated station 2311: (ANDERSON) → (ANDERSON)
✅ Updated station 2328: (OELRICH) → (OELRICH)
✅ Updated station 2332: (SASKUM) → (SASKUM)
✅ Updated station 11001: (NORTH SLOQUET) → (NORTH SLOQUET)
✅ Updated station 12018: (NARROWS INLET) → (NARROWS INLET)
✅ Updated station 12337: (INKANEEP EXT) → (INKANEEP EXT)
✅ Updated station 1

In [10]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {new_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 1694, TUMBLER HUB the names match 
✅ Station 1727, EUTSUK the names match 
✅ Station 1769, PLACE LAKE the names match 
✅ Station 1891, SALMON ARM the names match 
✅ Station 1908, REVELSTOKE the names match 
✅ Station 1926, FALLS CREEK the names match 
✅ Station 1927, HOWSER the names match 
✅ Station 1934, EIGHT MILE the names match 
✅ Station 1940, CRESTON the names match 
✅ Station 1957, TOBY HUB the names match 
✅ Station 2236, Bigattini the names match 
✅ Station 2302, TURNER _Tweedsmuir the names match 
✅ Station 2304, GOLDSTREAM 2 the names match 
✅ Station 2311, ANDERSON the names match 
✅ Station 2328, OELRICH the names match 
✅ Station 2332, SASKUM the names match 
✅ Station 11001, NORTH SLOQUET the names match 
✅ Station 12018, NARROWS INLET the names match 
✅ Station 12337, INKANEEP EXT the names match 
✅ Station 12524, APEX EXT the names match 
✅ Station 12622, MERRITT 2 HUB the names match 
✅ Station 12350, 85th Avenue the names match 
✅ Station 12343, Attachie F